# R-GCN Knowledge Graph Training — Google Colab

**Phase 3 of the NeuroVLM GNN experiment.**

Colab version of `rgcn_kg.ipynb`. Trains on a free T4 GPU (or A100 with Colab Pro).

| Step | What happens |
|---|---|
| Setup | Clone repo · install deps · mount Google Drive · configure GPU |
| 7 | Load unified KG · assign contiguous entity/relation integers · 85/7.5/7.5 stratified split · filtered negative sampling |
| 8 | Build R-GCN (2 × RGCNConv, basis decomposition) + DistMult decoder |
| 9 | Train with BCE loss · early stopping on val MRR · Hits@1/3/10 monitoring |
| 10 | Extract entity + relation embeddings · save with ID maps · nearest-neighbour spot-check |

---

### Before you start
1. **Runtime → Change runtime type → A100 GPU** (recommended) or T4.
2. Upload your two data files to Google Drive:
   ```
   MyDrive/neurovlm/data/unified_kg_nodes.parquet
   MyDrive/neurovlm/data/unified_kg_edges.parquet
   ```
3. Run all cells top-to-bottom.

### Resuming after a session crash
If Colab kills your session mid-training, just **re-run all cells from the top**.
The trainer automatically detects `resume_rgcn.pt` on Drive and picks up from the last saved epoch.

## 0 — Runtime check

In [ ]:
import subprocess, sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to Runtime → Change runtime type → select A100 GPU, then re-run."
    )
print(gpu_info.stdout)

## 1 — Install dependencies

In [ ]:
import torch
cuda_tag = 'cu' + torch.version.cuda.replace('.', '')
torch_tag = torch.__version__.split('+')[0]
print(f'torch={torch_tag}  cuda={cuda_tag}')

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyarrow', 'pandas', 'matplotlib'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric'], check=True)

pyg_url = f'https://data.pyg.org/whl/torch-{torch_tag}+{cuda_tag}.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_scatter', 'torch_sparse', '-f', pyg_url], check=True)

print('Dependencies installed.')

## 2 — Clone the repo

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/neurovlm')

if not REPO_DIR.exists():
    !git clone --branch neurovlm_gnn https://github.com/neurovlm/neurovlm.git {REPO_DIR}
else:
    print('Repo already cloned — pulling latest changes.')
    !git -C {REPO_DIR} pull

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print('sys.path entry:', src_dir)

In [ ]:
# Safety patch: lazy-import neurovlm.__init__ so nilearn is not required
init_path = REPO_DIR / 'src' / 'neurovlm' / '__init__.py'
init_path.write_text(
    'from __future__ import annotations\n\n\n'
    'def __getattr__(name: str):\n'
    '    if name == "NeuroVLM":\n'
    '        from .core import NeuroVLM\n'
    '        return NeuroVLM\n'
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n'
)
print('Patched:', init_path)

## 3 — Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT     = Path('/content/drive/MyDrive/neurovlm')
DATA           = DRIVE_ROOT / 'data'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'rgcn'
RESUME_PATH    = CHECKPOINT_DIR / 'resume_rgcn.pt'   # auto-saved every 5 epochs
EMB_PATH       = DRIVE_ROOT / 'embeddings' / 'entity_embeddings_v2.pt'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EMB_PATH.parent.mkdir(parents=True, exist_ok=True)

for f in [DATA / 'unified_kg_nodes.parquet', DATA / 'unified_kg_edges.parquet']:
    if not f.exists():
        raise FileNotFoundError(
            f'{f} not found.\nUpload the parquet files to MyDrive/neurovlm/data/ first.'
        )
    print('Found:', f)

if RESUME_PATH.exists():
    print(f'\nResume checkpoint found: {RESUME_PATH}')
    print('Training will continue from the last saved epoch.')
else:
    print('\nNo resume checkpoint — training will start from epoch 1.')

---
## Step 7 — Load KG and Prepare Training Data

In [ ]:
import gc
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

from neurovlm.gnn.kg_data import load_kg, KGSplits

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch  : {torch.__version__}', flush=True)
print(f'device : {device}', flush=True)

kg = load_kg(
    nodes_path=DATA / 'unified_kg_nodes.parquet',
    edges_path=DATA / 'unified_kg_edges.parquet',
)
print(f'KG loaded: {kg.num_entities:,} entities, {kg.num_relations} relation types, {len(kg.triples):,} triples.', flush=True)

In [ ]:
print(f'Entities  : {kg.num_entities:,}')
print(f'Relations : {kg.num_relations}')
print(f'Triples   : {len(kg.triples):,}')
print()
print('Relation type → integer index:')
for rel, idx in sorted(kg.relation_to_idx.items(), key=lambda x: x[1]):
    count = (kg.triples[:, 1] == idx).sum().item()
    print(f'  {idx}  {rel:<35}  {count:>7,} triples')

In [ ]:
splits = KGSplits.from_kg(kg, train_frac=0.85, val_frac=0.075, seed=42)

print(f'train edge_index : {splits.train_edge_index.shape}')
print(f'train edge_type  : {splits.train_edge_type.shape}')

In [ ]:
# Relation-frequency weights for weighted BCE loss
triple_rel  = kg.triples[:, 1]
counts      = torch.bincount(triple_rel, minlength=kg.num_relations).float()
raw_weights = (counts.sum() / kg.num_relations) / counts
rel_weights = (raw_weights / raw_weights.mean()).clamp(max=10.0)

print('Relation weights (mean-normalised, cap=10):')
for idx, name in sorted(kg.idx_to_relation.items()):
    print(f'  {idx}  {name:<35}  count={int(counts[idx]):>9,}  weight={rel_weights[idx]:.3f}')

In [ ]:
# Stash the small lookups we need, then free the raw triples tensor.
# splits.kg still holds a reference to kg, but deleting the local variable
# here avoids any accidental re-use of the full 360 MB triples tensor.
num_entities    = kg.num_entities
num_relations   = kg.num_relations
idx_to_entity   = kg.idx_to_entity
idx_to_relation = kg.idx_to_relation

del kg
gc.collect()
print('Local kg reference freed.', flush=True)

In [ ]:
# Verify stratification
for split_name, triples in [('train', splits.train_triples),
                              ('val',   splits.val_triples),
                              ('test',  splits.test_triples)]:
    rels_present = sorted(triples[:, 1].unique().tolist())
    print(f'{split_name:5s}  {len(triples):>7,} triples  relations present: {rels_present}')

In [ ]:
from torch.utils.data import DataLoader
from neurovlm.gnn.kg_data import kg_collate_fn

train_ds = splits.train_dataset(neg_ratio=10)
loader_preview = DataLoader(train_ds, batch_size=4, shuffle=False, collate_fn=kg_collate_fn)
batch = next(iter(loader_preview))

print('positives shape:', batch['positives'].shape)
print('negatives shape:', batch['negatives'].shape)
s, r, o = batch['positives'][0].tolist()
print(f"Example: {idx_to_entity[s]}  --[{idx_to_relation[r]}]-->  {idx_to_entity[o]}")

---
## Step 8 — Model Setup

In [ ]:
from neurovlm.gnn.rgcn import RGCNLinkPredictor

model = RGCNLinkPredictor(
    num_entities=num_entities,
    num_relations=num_relations,
    emb_dim=256,
    num_bases=None,
    num_layers=2,
    dropout=0.1,
)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total parameters: {n_params:,}')

In [ ]:
# Sanity-check: encode a 50k-edge sample (not the full 12.8M graph)
SAMPLE = 50_000
idx = torch.randperm(splits.train_edge_index.shape[1])[:SAMPLE]
sample_edge_index = splits.train_edge_index[:, idx]
sample_edge_type  = splits.train_edge_type[idx]

with torch.no_grad():
    emb = model.encode(sample_edge_index, sample_edge_type)
    print('entity_emb shape :', emb.shape)
    pos    = splits.train_triples[:4]
    scores = model.score(emb, pos[:, 0], pos[:, 1], pos[:, 2])
    print('pos scores       :', scores.round(decimals=4))

del emb, sample_edge_index, sample_edge_type, idx
gc.collect()
print('Sanity check passed.', flush=True)

---
## Step 9 — Training

**Why the batch/steps config:**
- 12.8M training triples ÷ batch_size=1024 = 12,545 batches/epoch → **993s/epoch** (what you saw)
- With `batch_size=4096` + `max_steps_per_epoch=1000`: 1,000 batches/epoch → **~40s/epoch**
- Each epoch still covers 4M triples (31% of train set); full coverage across ~3 epochs

**Checkpointing:**
- `best_rgcn.pt` — saved whenever validation MRR improves
- `resume_rgcn.pt` — saved every 5 epochs with full state (weights, optimizer, scheduler, history, epoch)

**To resume after a crash:** re-run all cells from the top — the trainer auto-detects `resume_rgcn.pt`.

**Progress printing:** loss every epoch · MRR + Hits@k every 5 epochs

In [ ]:
from neurovlm.gnn.kg_train import RGCNTrainer

trainer = RGCNTrainer(
    model=model,
    splits=splits,
    lr=1e-3,
    weight_decay=1e-4,
    n_epochs=500,
    batch_size=4096,            # was 1024 — larger batch = fewer steps, better GPU util
    neg_ratio=10,
    eval_batch_size=64,
    val_interval=5,
    patience=75,
    lr_patience=10,
    lr_factor=0.5,
    graph_sample_size=200_000,
    device='auto',
    checkpoint_dir=CHECKPOINT_DIR,
    verbose=True,
    relation_weights=rel_weights,
    print_interval=1,
    resume_checkpoint_interval=5,
    resume_from=RESUME_PATH,
    max_steps_per_epoch=1000,   # cap: 1000 × 4096 = 4M triples/epoch (~40s on A100)
)

In [ ]:
trainer.fit()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(trainer.history['train_loss'])
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

val_epochs = list(range(5, 5 * len(trainer.history['val_mrr']) + 1, 5))
ax2.plot(val_epochs, trainer.history['val_mrr'],    label='MRR')
ax2.plot(val_epochs, trainer.history['val_hits@1'], label='Hits@1', linestyle='--')
ax2.plot(val_epochs, trainer.history['val_hits@3'], label='Hits@3', linestyle='--')
ax2.plot(val_epochs, trainer.history['val_hits@10'],label='Hits@10',linestyle='--')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title('Validation Metrics (filtered)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DRIVE_ROOT / 'rgcn_training_curves.png', dpi=150)
plt.show()
print(f'Best val MRR: {trainer._best_mrr:.4f}')

---
## Test-set Evaluation

In [ ]:
test_metrics = trainer.evaluate_test()

print('\nTest results (filtered ranking):')
for k, v in test_metrics.items():
    print(f'  {k:<10} {v:.4f}')

---
## Step 10 — Extract and Save Embeddings

In [ ]:
trainer.save_embeddings(EMB_PATH)
print(f'Embeddings saved to {EMB_PATH}')

In [ ]:
import json as _json

emb_payload  = torch.load(EMB_PATH, weights_only=True)
entity_emb   = emb_payload['entity_embeddings']
relation_emb = emb_payload['relation_embeddings']

meta_path = EMB_PATH.with_suffix('').with_suffix('.pt.meta.json')
with open(meta_path) as fh:
    meta = _json.load(fh)
idx_to_relation = {int(k): v for k, v in meta['idx_to_relation'].items()}
idx_to_entity   = {int(k): v for k, v in meta['idx_to_entity'].items()}

nodes_df   = pd.read_parquet(DATA / 'unified_kg_nodes.parquet')
id_to_name = nodes_df.set_index('canonical_id')['name'].to_dict()

print('entity_embeddings  :', entity_emb.shape)
print('relation_embeddings:', relation_emb.shape)
print()
print('Relation embedding norms:')
for idx, name in sorted(idx_to_relation.items()):
    norm = relation_emb[idx].norm().item()
    print(f'  {idx}  {name:<35}  norm={norm:.4f}')

In [ ]:
def show_neighbours(query: str, k: int = 10):
    neighbours = trainer.nearest_neighbours(query, k=k, entity_emb=entity_emb)
    query_name = id_to_name.get(query, query)
    print(f"\nNearest neighbours for '{query_name}' ({query}):")
    for sim, eid in neighbours:
        name = id_to_name.get(eid, eid)
        print(f'  {sim:.4f}  {name:<45}  ({eid})')

show_neighbours('D006624')  # Hippocampus
show_neighbours('D000544')  # Alzheimer Disease
show_neighbours('D000679')  # Amygdala

In [ ]:
# UMAP visualisation (optional — pip install umap-learn)
try:
    import umap
    import numpy as np

    nodes_df = pd.read_parquet(DATA / 'unified_kg_nodes.parquet')
    type_map = nodes_df.set_index('canonical_id')['node_type'].to_dict()

    emb_np  = F.normalize(entity_emb, dim=-1).cpu().numpy()
    reducer = umap.UMAP(n_components=2, metric='cosine', random_state=42, n_neighbors=15)
    coords  = reducer.fit_transform(emb_np)

    node_types   = [type_map.get(idx_to_entity[i], 'other') for i in range(num_entities)]
    unique_types = sorted(set(node_types))
    color_map    = {t: plt.cm.tab10(i / len(unique_types)) for i, t in enumerate(unique_types)}
    colors       = [color_map[t] for t in node_types]

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=1, alpha=0.4, linewidths=0)
    handles = [plt.Line2D([0], [0], marker='o', color='w',
                          markerfacecolor=color_map[t], markersize=8, label=t)
               for t in unique_types]
    ax.legend(handles=handles, title='node_type', bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.set_title('UMAP of R-GCN entity embeddings (cosine)')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(DRIVE_ROOT / 'rgcn_umap.png', dpi=150, bbox_inches='tight')
    plt.show()

except ImportError:
    print('umap-learn not installed — skipping.\n!pip install umap-learn')

---
## Output Files

All outputs saved to `MyDrive/neurovlm/`:

| File | Description |
|---|---|
| `checkpoints/rgcn/best_rgcn.pt` | Best model checkpoint (saved on every MRR improvement) |
| `checkpoints/rgcn/resume_rgcn.pt` | Full resume state saved every 5 epochs |
| `embeddings/entity_embeddings_v2.pt` | Entity + relation embeddings with ID maps |
| `rgcn_training_curves.png` | Loss + MRR training curves |
| `rgcn_umap.png` | UMAP of entity embedding space (if umap-learn installed) |